# Remaining Useful Life ($\textit{RUL}$) estimation of turbofan engines via Federated Averaging

This notebook contains code for estimating the Remaining Useful Life ($\textit{RUL}$) of turbofan engines experiencing a determined flight lengh (flight class). This case study makes use of the **new** C-MAPSS (Commercial Modular Aero-Propulsion System Simulation) dataset from NASA for aircraft engines.  More details about the generation process can be found at https://www.mdpi.com/2306-5729/6/1/5. 

This **new** C-MAPSS dataset comprises run-to-failure data of multiple engine units located at different $\textit{DS}$ sets. The degradation of each engine unit was simulated considering real operating conditions from a real jet in multiple flights. Flights are classified according their lenght: short-length flights (i.e., flight class 1), medium-length flights (i.e., flight class 2), or long-length flights (i.e., flight class 2).

| Flight Class   | Flight Length [h]
| :-----------:  | :-----------:    
| 1              |    1 to 3        
| 2              |    3 to 5        
| 3              |    5 to 7  

This notebook reused the inception-based CNN network from (https://doi.org/10.36001/phmconf.2021.v13i1.3109) and its pre-processing precedures to estimate the $\textit{RUL}$ of engines experiencing a determined flight class (FC) in a single party. 

This data partitioning is then used to simulate a collaborative prognostics problem in which three parties, each storing data from a determined FC, learn from each others without sharing raw data. Parties aggregate their knowledge using Federated Avergaging FedAvg algorithm to construct a federated model.

To estimate the $\textit{RUL}$ of all available turbofan engines of a determined FC in $\textit{DS}$ sets, you must define the FC number in the hyperparameters section. You get a dataset for each class by previously executing:

- 1) Spliting a given dataset by Flight Class.ipynb
- 2) Concatenating datasets by Flight Class.ipynb


## Before we start

Before we start, please run the following to make sure that your environment is
correctly setup. If you don't see a greeting, please refer to the
[Installation](../install.md) guide for instructions. 

## Install Libraries

In [ ]:
#@test {"skip": true}

#!pip install --quiet --upgrade tensorflow-federated

## Load Libraries

In [ ]:
# packages
import os
import h5py
import time
import random
import itertools
import collections
import tensorflow as tf
import tensorflow_federated as tff
import matplotlib.pyplot as plt
from matplotlib import gridspec
import pandas as pd
import numpy as np
from time import gmtime, strftime
import random
import keras.backend as K
from tensorflow import keras
from keras import layers
from keras import models
from keras import regularizers
from keras.models import model_from_json
from sklearn.utils import shuffle
from pandas import DataFrame

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression
from itertools import product
from sklearn.decomposition import PCA
# import tensorflow_addons as tfa

tff.federated_computation(lambda: 'Hello, World!')()

# EVALUATION modules
from scipy.stats import spearmanr

params = {'legend.fontsize': 8,
          'figure.figsize': (9,6),
         'axes.labelsize': 20,
         'axes.titlesize':20,
         'xtick.labelsize':'xx-large',
         'axes.linewidth' : 2,
         'ytick.labelsize':'xx-large'}

plt.rcParams.update(params)

## Hyperparameters

In [ ]:
# Flight Class FC
FC = 3
DOWNSAMPLING_STEP=[2,2,8]
K = 300
# LAYERS = [64,32]
LR = 0.01
BATCH_SIZE = 128
EPOCHS = 200

# PRE-PROCESSING
NUM_EPOCHS = 5
SHUFFLE_BUFFER = 1000
PREFETCH_BUFFER = 100
WINDOW_LEN = 30
stride = 8

PIECE_WISE=True

# SCALERS
scaler_X = StandardScaler()
scaler_Y = MinMaxScaler(feature_range=(0,1))

seed=571

np.random.seed(seed)

## Preparing the input data

Let's start with the data. Federated learning requires a federated data set,
i.e., a collection of data from multiple users. Federated data is typically
non-[i.i.d.](https://en.wikipedia.org/wiki/Independent_and_identically_distributed_random_variables),
which poses a unique set of challenges.


In [ ]:
def normalize_data(x, lb, ub, max_v=1.0, min_v=-1.0):

    # Set-up
    if (lb == []) & (ub == []):
        # OPTION 1:
        ub = x.max(0)
        lb = x.min(0)
    
        # OPTION 2:
        #ub = np.percentile(x, 99.9, axis=0, keepdims=True)
        #lb = np.percentile(x, 0.1, axis=0, keepdims=True)
    
    ub.shape = (1,-1)
    lb.shape = (1,-1)           
    max_min = max_v - min_v
    delta = ub-lb

    # Compute
    x_n = max_min * (x - lb) / delta + min_v
    if 0 in delta:
        idx = np.ravel(delta == 0)
        x_n[:,idx] = x[:,idx] - lb[:, idx]

    return x_n, lb, ub 


def split_dataset(dataset, split=4): 
    '''
    Split 'dataset' in tree pieces: w, x_s, theta
    '''
    w = dataset[:,:split]
    x_s = dataset[:,split:20]
    theta = dataset[:,-10:]
    tmp = theta[:,6]
    tmp.shape = (-1,1)
    
    return w, x_s, tmp

def extract_units_ds(id_en, ds, units):
    '''
    Creates a subset with only id_en units for ds
    '''
    
    # Set-up
    ds_sub = []
    units_unique = np.unique(units)

    # Process
    for i in units_unique:
        if i in id_en:
            idx = np.ravel(units==i)
            ds_sub.append(ds[idx,:])           
    
    return np.concatenate(ds_sub, axis=0)

def create_lag_data(w, x_s, theta, y, Units, Cycles, stride=1):
    # Set-up
    W, X, T, Y, U, C, _W, _X = [], [], [], [], [], [], [], []
    
    # Loop over units and then within the units
    units = np.unique(Units)
    for k in units:
        unit = np.ravel(Units == k)
        w_unit = w[unit,:]
        x_s_unit = x_s[unit,:]
        T_unit = theta[unit,:]
        Y_unit = y[unit,:]
        U_unit = Units[unit,:]
        C_unit = Cycles[unit,:]
        dim = w_unit.shape[0]
        for i in range(dim-1): 
            X.append(x_s_unit[i + stride, :])          # X  or X_
            W.append(w_unit[i + stride, :])            # W  or W_
            T.append(T_unit[i + stride, :])            # T  or T_
            Y.append(Y_unit[i + stride, :])            # T  or T_
            U.append(U_unit[i + stride, :])            # U  or U_
            C.append(C_unit[i + stride, :])            # C  or C_
            _X.append(x_s_unit[i, :])                  # _X or X
            _W.append(w_unit[i, :])                  # _X or X
            
    return np.array(W), np.array(X), np.array(T), np.array(Y), np.array(U), np.array(C), np.array(_X), np.array(_W)



# np.array(X_out), np.array(W_out), np.array(Y_out), np.array(T_out), np.array(U_out), np.array(C_out), np.array(HI_out)

def split_sequences(input_data, sequence_length, stride = 1, option = None):
    """
     
    """
    X = list()
    
    for i in range(0,len(input_data),stride):
        # find the end of this pattern
        end_ix = i + sequence_length
        
        # check if we are beyond the dataset
        if end_ix > len(input_data):
            break
        
        # gather input and output parts of the pattern
        if option=='last':
            seq_x = input_data[end_ix-1, :]
        elif option=='next':
            seq_x = input_data[end_ix, :]
        else:
            seq_x = input_data[i:end_ix, :]
        X.append(seq_x)
    
    return np.array(X)


## Sequence Generator

In [ ]:
def sequence_generator(input_data, units, cycles, sequence_length=10,stride = 1, option=None):
    """
     # Generates dataset with windows of sequence_length      
    """  
    X = list()
    unit_num=[]
    c_num =[]
    for i, elem_u in enumerate(list(np.unique(units))):
        mask = np.ravel(units==elem_u)
        c_mask = cycles[mask]
        x_unit = input_data[mask]
        for j in np.unique(c_mask):
            mask = np.ravel(c_mask==j)
            seq_x_u = split_sequences(x_unit[mask],sequence_length, stride, option)
            X.append(seq_x_u)
            unit_num.extend(np.ones(len(seq_x_u),dtype = int)*elem_u)
            c_num.extend(np.ones(len(seq_x_u),dtype = int)*j)
    
    return np.vstack(X),np.array(unit_num).reshape(-1,1),np.array(c_num).reshape(-1,1)


def sequence_generator_per_unit(input_data, units, cycles, sequence_length=10, stride =1,option=None):
    """
     # Generates dataset with windows of sequence_length      
    """  
    X = list()
    unit_num=[]
    c_num =[]
    for i, elem_u in enumerate(list(np.unique(units))):
        mask = np.ravel(units==elem_u)
        x_unit = input_data[mask]
        seq_x_u = split_sequences(x_unit,sequence_length, stride, option)
        X.append(seq_x_u)
        unit_num.extend(np.ones(len(seq_x_u),dtype = int)*elem_u)
        c_num.append(split_sequences(cycles[mask],sequence_length, stride, option))
    
    return np.vstack(X),np.array(unit_num).reshape(-1,1),np.vstack(c_num)

## Load Data

In [ ]:
def correctRUL(hi, RUL):
    if hi == 1:
      return -1
    else:
      return RUL
def correctMaxRUL(hi, RUL,MAX_RUL):
    if hi == 1:
      return MAX_RUL
    else:
      return RUL

In [ ]:
def load_data_from_flight_classes():
    for FC in range(0, 3):
            # Load data DEV
            with h5py.File("FC"+str(FC+1)+"/FC"+str(FC+1)+'_test'+".h5", 'r') as hdf:
                # Development set
                W_test = np.array(hdf.get('W_test'), dtype='float16')             # W
                X_s_test = np.array(hdf.get('X_s_test'), dtype='float16')         # X_s
                Y_test = np.array(hdf.get('Y_test'), dtype='float16')             # RUL                  
                A_test = np.array(hdf.get('A_test'), dtype='float16')
                
                W_test = W_test[::DOWNSAMPLING_STEP[FC],:]
                X_s_test = X_s_test[::DOWNSAMPLING_STEP[FC],:] 
                Y_test = Y_test[::DOWNSAMPLING_STEP[FC]]
                A_test = A_test[::DOWNSAMPLING_STEP[FC],:]

                # Varnams
                W_var = np.array(hdf.get('W_var'))
                X_s_var = np.array(hdf.get('X_s_var'))  
                X_v_var = np.array(hdf.get('X_v_var')) 
                T_var = np.array(hdf.get('T_var'))
                A_var = np.array(hdf.get('A_var'))

                # from np.array to list dtype U4/U5
                W_var = list(np.array(W_var, dtype='U20'))
                X_s_var = list(np.array(X_s_var, dtype='U20'))  
                X_v_var = list(np.array(X_v_var, dtype='U20')) 
                T_var = list(np.array(T_var, dtype='U20'))
                A_var = list(np.array(A_var, dtype='U20'))
            
            if FC==0:
                W_test_aux = W_test
                X_s_test_aux = X_s_test
                Y_test_aux = Y_test
                A_test_aux = A_test
            if FC!=0:
                W_test_aux = np.concatenate((W_test_aux, W_test), axis=0)  
                X_s_test_aux = np.concatenate((X_s_test_aux, X_s_test), axis=0)
                Y_test_aux = np.concatenate((Y_test_aux, Y_test), axis=0) 
                A_test_aux = np.concatenate((A_test_aux, A_test), axis=0)
                
    units_test=A_test_aux[:,0].reshape(-1,1)
    cycles_test=A_test_aux[:,1].reshape(-1,1)
    hi_test = A_test_aux[:,-1]

    if PIECE_WISE==True:
        df_hs_unit_test = DataFrame({'unit': units_test.reshape(-1).astype(int),'RUL': Y_test_aux.reshape(-1), 'hi': hi_test.reshape(-1)})
        df_hs_unit_test['RUL']=df_hs_unit_test.apply(lambda row: correctRUL(row['hi'],row['RUL']), axis=1)

        pd_aux=DataFrame(df_hs_unit_test.groupby('unit')['RUL'].max()).reset_index()
        df_hs_unit_test['RUL']=df_hs_unit_test.apply(lambda row: correctMaxRUL(row['hi'],row['RUL'], float(pd_aux.iloc[pd_aux.index[pd_aux['unit'] == row['unit']]]['RUL'])), axis=1)
        Y_test_aux=df_hs_unit_test['RUL'].to_numpy().reshape(len(df_hs_unit_test),1)

    X_s_test_aux = np.concatenate((X_s_test_aux, W_test_aux), axis=1)
    X_s_test_aux = scaler_X.fit_transform(X_s_test_aux)      
    Y_test_aux = scaler_Y.fit_transform(Y_test_aux)
                
    X_windows_test, U_windows_test,C_windows_test=sequence_generator_per_unit(X_s_test_aux,units_test,cycles_test,sequence_length=WINDOW_LEN,stride = stride)
    Y_windows_test,_,_=sequence_generator_per_unit(Y_test_aux,units_test,cycles_test,sequence_length=WINDOW_LEN,option='last',stride = stride)
    shuffle_idx = np.random.permutation(X_windows_test.shape[0])
    return X_windows_test[shuffle_idx], Y_windows_test[shuffle_idx]

def load_data_from_flight_class(FC):
    # Load data DEV
    with h5py.File("FC"+str(FC)+"/FC"+str(FC)+'_dev'+".h5", 'r') as hdf:
                # Development set
                W_train = np.array(hdf.get('W_dev'), dtype='float16')             # W
                X_s_train = np.array(hdf.get('X_s_dev'), dtype='float16')         # X_s
                X_v_train = np.array(hdf.get('X_v_dev'), dtype='float16')         # X_v
                T_train = np.array(hdf.get('T_dev'), dtype='float16')             # T
                Y_train = np.array(hdf.get('Y_dev'), dtype='float16')             # RUL  
                A_train = np.array(hdf.get('A_dev'), dtype='float16')
                
                W_train = W_train[::DOWNSAMPLING_STEP[FC-1],:]
                X_s_train = X_s_train[::DOWNSAMPLING_STEP[FC-1],:] 
                X_v_train = X_v_train[::DOWNSAMPLING_STEP[FC-1],:] 
                T_train = T_train[::DOWNSAMPLING_STEP[FC-1],:]
                Y_train = Y_train[::DOWNSAMPLING_STEP[FC-1],:]
                A_train = A_train[::DOWNSAMPLING_STEP[FC-1],:]
                
                # Varnams
                W_var = np.array(hdf.get('W_var'))
                X_s_var = np.array(hdf.get('X_s_var'))  
                X_v_var = np.array(hdf.get('X_v_var')) 
                T_var = np.array(hdf.get('T_var'))
                A_var = np.array(hdf.get('A_var'))
                
                    # from np.array to list dtype U4/U5
                W_var = list(np.array(W_var, dtype='U20'))
                X_s_var = list(np.array(X_s_var, dtype='U20'))  
                X_v_var = list(np.array(X_v_var, dtype='U20')) 
                T_var = list(np.array(T_var, dtype='U20'))
                A_var = list(np.array(A_var, dtype='U20'))
                

    units_train=A_train[:,0].reshape(-1,1)
    cycles_train=A_train[:,1].reshape(-1,1)
    fc_train = A_train[:,2].reshape(-1,1)
    hi_train = A_train[:,-1]

    # RUL Piece-wise correction
    if PIECE_WISE==True:
        df_hs_unit_train = DataFrame({'unit': units_train.reshape(-1).astype(int),'RUL': Y_train.reshape(-1), 'hi': hi_train.reshape(-1)})
        df_hs_unit_train['RUL']=df_hs_unit_train.apply(lambda row: correctRUL(row['hi'],row['RUL']), axis=1)

        pd_aux=DataFrame(df_hs_unit_train.groupby('unit')['RUL'].max()).reset_index()
        df_hs_unit_train['RUL']=df_hs_unit_train.apply(lambda row: correctMaxRUL(row['hi'],row['RUL'], float(pd_aux.iloc[pd_aux.index[pd_aux['unit'] == row['unit']]]['RUL'])), axis=1)
        Y_train=df_hs_unit_train['RUL'].to_numpy().reshape(len(df_hs_unit_train),1)
    
    # Max Min SCALE

    # scaler_X = MinMaxScaler(feature_range=(-1,1)
    
    X_s_train = np.concatenate((X_s_train, W_train), axis=1)
    X_s_train = scaler_X.fit_transform(X_s_train)
    
    print("XS_train",X_s_train.shape)
    print(units_train.shape)


    Y_train = scaler_Y.fit_transform(Y_train)
    
    # Downsampling 0.1Hz
    
    X_windows, U_windows, C_windows=sequence_generator(X_s_train,units_train,cycles_train,sequence_length=WINDOW_LEN,stride = stride)
    Y_windows,_,_=sequence_generator(Y_train,units_train,cycles_train,sequence_length=WINDOW_LEN,option='last',stride = stride)
    
    shuffle_idx = np.random.permutation(X_windows.shape[0])
    training_idx, val_idx = shuffle_idx[X_windows.shape[0]//10:],shuffle_idx[:X_windows.shape[0]//10]

    return X_windows[training_idx], Y_windows[training_idx], X_windows[val_idx], Y_windows[val_idx]

In [ ]:
x, y = load_data_from_flight_classes()
y[0]

In [ ]:
x,y,z,w=load_data_from_flight_class(1)
y[0]

### Training and Validation data

In [ ]:
flight_classes = [1,2,3]

def preprocess(dataset):

  def batch_format_fn(element):
    """Flatten a batch `pixels` and return the features as an `OrderedDict`."""
    return collections.OrderedDict(
        x=tf.reshape(element['x'], [-1, WINDOW_LEN, 18]),
        y=tf.reshape(element['y'], [-1, 1]))

  return dataset.shuffle(SHUFFLE_BUFFER).batch(
      BATCH_SIZE, drop_remainder=True).map(batch_format_fn).prefetch(PREFETCH_BUFFER)
  
def create_tf_dataset_from_dataframe(id):
    x_train, y_train, x_test, y_test = load_data_from_flight_class(id)
    """Converts a Pandas DataFrame to a tf.data.Dataset."""
    dataset_train = tf.data.Dataset.from_tensor_slices(
        {"x": tf.cast(x_train, tf.float16), "y": tf.cast(y_train, tf.float16)}
    )
    dataset_test = tf.data.Dataset.from_tensor_slices(
        {"x": tf.cast(x_test, tf.float16), "y": tf.cast(y_test, tf.float16)}
    )
    return preprocess(dataset_train), preprocess(dataset_test)

def create_tf_dataset_from_flight_classes():
    x, y = load_data_from_flight_classes()
    """Converts a Pandas DataFrame to a tf.data.Dataset."""
    dataset = tf.data.Dataset.from_tensor_slices(
        {"x": tf.cast(x, tf.float16), "y": tf.cast(y, tf.float16)}
    )
    return preprocess(dataset)

### Preprocessing the input data
Since the data is already a `tf.data.Dataset`,  preprocessing can be accomplished using Dataset transformations. Here, we flatten the `28x28` images
into `784`-element arrays, shuffle the individual examples, organize them into batches, and rename the features
from `pixels` and `label` to `x` and `y` for use with Keras. We also throw in a
`repeat` over the data set to run several epochs.

In [ ]:
federated_train_data, federated_eval_data = create_tf_dataset_from_dataframe(1)

sample_batch_train = tf.nest.map_structure(lambda x: x.numpy(),
                                     next(iter(federated_train_data)))

sample_batch_train

Let's verify this worked.

In [ ]:
federated_test_data = (create_tf_dataset_from_flight_classes())


sample_batch_test = tf.nest.map_structure(lambda x: x.numpy(),
                                     next(iter(federated_test_data)))

sample_batch_test

In [ ]:
def make_federated_data(client_ids):
  federated_train = []
  federated_test = []
  # Use a for loop to iterate and capture only the first element
  for i in client_ids:  # Assuming the range of indices is 0 to 9
      dataset_train, dataset_test = create_tf_dataset_from_dataframe(i)  # Unpack and ignore the second value
      federated_train.append((dataset_train))
      federated_test.append((dataset_test))
  return federated_train, federated_test

In [ ]:

federated_train_data,  federated_eval_data = make_federated_data(flight_classes)

print(f'Number of train datasets: {len(federated_train_data)}')
print(f'Number of test datasets: {len(federated_eval_data)}')
print(f'First train dataset: {federated_train_data[0]}')
print(f'First test dataset: {federated_eval_data[0]}')

## Creating a model with Keras

If you are using Keras, you likely already have code that constructs a Keras
model. Here's an example of a simple model that will suffice for our needs.

### Model Inception

In [ ]:

def inception2D(t=64,
      feature_X_in=18,
      feature_out_size=1):
    
    '''
    useH: if True, use H as input
        [X,W,H] -> Y 
    else:
        [X,W] -> Y
    '''

    x=layers.Input(shape=(t,feature_X_in,1),name="x")
          
    layer_1 = tf.keras.layers.Conv2D(10, (3,3), padding='same', activation='relu')(x)

    layer_2 = tf.keras.layers.Conv2D(10, (5,5), padding='same', activation='relu')(x)

    layer_3 = tf.keras.layers.MaxPooling2D(3, strides=(1,1), padding='same')(x)
    layer_3 = tf.keras.layers.Conv2D(10, (1,1), padding='same', activation='relu')(layer_3)

    mid_1 = tf.keras.layers.concatenate([layer_1, layer_2, layer_3], axis = 3)

    ### 2nd Module
    layer_4 = tf.keras.layers.Conv2D(10, (1,1), padding='same', activation='relu')(mid_1)
    layer_4 = tf.keras.layers.Conv2D(10, (3,3), padding='same', activation='relu')(layer_4)

    layer_5 = tf.keras.layers.Conv2D(10, (1,1), padding='same', activation='relu')(mid_1)
    layer_5 = tf.keras.layers.Conv2D(10, (5,5), padding='same', activation='relu')(layer_5)

    layer_6 = tf.keras.layers.MaxPooling2D(1, strides=(1,1), padding='same')(mid_1)
    layer_6 = tf.keras.layers.Conv2D(10, (1,1), padding='same', activation='relu')(layer_6)

    mid_2 = tf.keras.layers.concatenate([layer_4, layer_5, layer_6], axis = 2)

    flat_1 = tf.keras.layers.Flatten()(mid_2)

    drop = tf.keras.layers.Dropout(.5)(flat_1)

    dense_1 = tf.keras.layers.Dense(256, activation='sigmoid')(drop)
    y = tf.keras.layers.Dense(feature_out_size, activation='relu')(dense_1)

    model = models.Model([x], y)

    return model

def inception1D(t=64,
      feature_X_in=18,
      feature_out_size=1):
    
    '''
    useH: if True, use H as input
        [X,W,H] -> Y 
    else:
        [X,W] -> Y
    '''

    x=layers.Input(shape=(t,feature_X_in),name="x")
    
      
    layer_1 = tf.keras.layers.Conv1D(10, 3, padding='same', activation='relu')(x)

    layer_2 = tf.keras.layers.Conv1D(10, 5, padding='same', activation='relu')(x)

    layer_3 = tf.keras.layers.MaxPooling1D(3, strides=1, padding='same')(x)
    layer_3 = tf.keras.layers.Conv1D(10, 1, padding='same', activation='relu')(layer_3)

    mid_1 = tf.keras.layers.concatenate([layer_1, layer_2, layer_3], axis = 2)

    ### 2nd Module
    layer_4 = tf.keras.layers.Conv1D(10, 1, padding='same', activation='relu')(mid_1)
    layer_4 = tf.keras.layers.Conv1D(10, 3, padding='same', activation='relu')(layer_4)

    layer_5 = tf.keras.layers.Conv1D(10, 1, padding='same', activation='relu')(mid_1)
    layer_5 = tf.keras.layers.Conv1D(10, 5, padding='same', activation='relu')(layer_5)

    layer_6 = tf.keras.layers.MaxPooling1D(1, strides=1, padding='same')(mid_1)
    layer_6 = tf.keras.layers.Conv1D(10, 1, padding='same', activation='relu')(layer_6)

    mid_2 = tf.keras.layers.concatenate([layer_4, layer_5, layer_6], axis = 2)

    flat_1 = tf.keras.layers.Flatten()(mid_2)

    drop = tf.keras.layers.Dropout(.5)(flat_1)

    dense_1 = tf.keras.layers.Dense(256, activation='sigmoid')(drop)
    y = tf.keras.layers.Dense(feature_out_size, activation='relu')(dense_1)

    model = models.Model([x], y)

    return model

**Note:** we do not compile the model yet. The loss, metrics, and optimizers are introduced later.

In order to use any model with TFF, it needs to be wrapped in an instance of the
`tff.learning.models.VariableModel` interface, which exposes methods to stamp the model's
forward pass, metadata properties, etc., similarly to Keras, but also introduces
additional elements, such as ways to control the process of computing federated
metrics. Let's not worry about this for now; if you have a Keras model like the
one we've just defined above, you can have TFF wrap it for you by invoking
`tff.learning.models.from_keras_model`, passing the model and a sample data batch as
arguments, as shown below.

In [ ]:
class RootMeanSquaredErrorLoss(tf.keras.losses.Loss):
    def __init__(self, name="root_mean_squared_error_loss"):
        super().__init__(name=name)

    def call(self, y_true, y_pred):
        error = y_true - y_pred
        squared_error = tf.square(error)
        mean_squared_error = tf.reduce_mean(squared_error)
        root_mean_squared_error = tf.sqrt(mean_squared_error)
        return root_mean_squared_error

In [ ]:
def model_fn():
  # We _must_ create a new model here, and _not_ capture it from an external
  # scope. TFF will call this within different graph contexts.
  keras_model = inception1D(WINDOW_LEN)
  return tff.learning.models.from_keras_model(
      keras_model,
      input_spec=federated_test_data.element_spec,
      loss=tf.keras.losses.Huber(),
      metrics=[tf.keras.metrics.MeanAbsoluteError()])

## Training the model on federated data

Now that we have a model wrapped as `tff.learning.models.VariableModel` for use with TFF, we
can let TFF construct a Federated Averaging algorithm by invoking the helper
function `tff.learning.algorithms.build_weighted_fed_avg`, as follows.

Keep in mind that the argument needs to be a constructor (such as `model_fn`
above), not an already-constructed instance, so that the construction of your
model can happen in a context controlled by TFF (if you're curious about the
reasons for this, we encourage you to read the follow-up tutorial on
[custom algorithms](custom_federated_algorithms_1.ipynb)).

One critical note on the Federated Averaging algorithm below, there are **2**
optimizers: a _client_optimizer_ and a _server_optimizer_. The
_client_optimizer_ is only used to compute local model updates on each client.
The _server_optimizer_ applies the averaged update to the global model at the
server. In particular, this means that the choice of optimizer and learning rate
used may need to be different than the ones you have used to train the model on
a standard i.i.d. dataset. We recommend starting with regular SGD, possibly with
a smaller learning rate than usual. The learning rate we use has not been
carefully tuned, feel free to experiment.

In [ ]:
fed_avg = tff.learning.algorithms.build_weighted_fed_avg(
    model_fn,
    client_optimizer_fn=tff.learning.optimizers.build_sgdm(learning_rate=LR))

What just happened? TFF has constructed a pair of *federated computations* and
packaged them into a `tff.templates.IterativeProcess` in which these computations
are available as a pair of properties `initialize` and `next`.

In a nutshell, *federated computations* are programs in TFF's internal language
that can express various federated algorithms (you can find more about this in
the [custom algorithms](custom_federated_algorithms_1.ipynb) tutorial). In this
case, the two computations generated and packed into `iterative_process`
implement [Federated Averaging](https://arxiv.org/abs/1602.05629).

It is a goal of TFF to define computations in a way that they could be executed
in real federated learning settings, but currently only local execution
simulation runtime is implemented. To execute a computation in a simulator, you
simply invoke it like a Python function. This default interpreted environment is
not designed for high performance, but it will suffice for this tutorial; we
expect to provide higher-performance simulation runtimes to facilitate
larger-scale research in future releases.

Let's start with the `initialize` computation. As is the case for all federated
computations, you can think of it as a function. The computation takes no
arguments, and returns one result - the representation of the state of the
Federated Averaging process on the server. While we don't want to dive into the
details of TFF, it may be instructive to see what this state looks like. You can
visualize it as follows.

In [ ]:
print(fed_avg.initialize.type_signature.formatted_representation())

While the above type signature may at first seem a bit cryptic, you can
recognize that the server state consists of a `global_model_weights` (the initial model parameters for MNIST that will be distributed to all devices), some empty parameters (like `distributor`, which governs the server-to-client communication) and a `finalizer` component. This last one governs the logic that the server uses to update its model at the end of a round, and contains an integer representing how many rounds of FedAvg have occurred.

Let's invoke the `initialize` computation to construct the server state.

The second of the pair of federated computations, `next`, represents a single
round of Federated Averaging, which consists of pushing the server state
(including the model parameters) to the clients, on-device training on their
local data, collecting and averaging model updates, and producing a new updated
model at the server.

Conceptually, you can think of `next` as having a functional type signature that
looks as follows.

```
SERVER_STATE, FEDERATED_DATA -> SERVER_STATE, TRAINING_METRICS
```

In particular, one should think about `next()` not as being a function that runs on a server, but rather being a declarative functional representation of the entire decentralized computation - some of the inputs are provided by the server (`SERVER_STATE`), but each participating device contributes its own local dataset.

Let's run a single round of training and visualize the results. We can use the
federated data we've already generated above for a sample of users.

In [ ]:
train_state = fed_avg.initialize()
result = fed_avg.next(train_state, federated_train_data)
train_state = result.state
train_metrics = result.metrics
print('round  1, metrics={}'.format(train_metrics))

In [ ]:
for i in federated_test_data:
    print(i)

In [ ]:
keras_model = inception1D(WINDOW_LEN)
keras_model.compile(
      loss=tf.keras.losses.Huber(),
      metrics=[tf.keras.metrics.MeanAbsoluteError()])
model_weights = fed_avg.get_model_weights(train_state)
model_weights.assign_weights_to(keras_model)
results  = keras_model.evaluate(federated_test_data)

Let's run a few more rounds. As noted earlier, typically at this point you would
pick a subset of your simulation data from a new randomly selected sample of
users for each round in order to simulate a realistic deployment in which users
continuously come and go, but in this interactive notebook, for the sake of
demonstration we'll just reuse the same users, so that the system converges
quickly.

In [ ]:


def keras_evaluate(state, round_num):
  # Take our global model weights and push them back into a Keras model to
  # use its standard `.evaluate()` method.
  keras_model = inception1D(WINDOW_LEN)
  keras_model.compile(
      loss=RootMeanSquaredErrorLoss(),
      metrics=[tf.keras.metrics.MeanAbsoluteError()])
  model_weights = fed_avg.get_model_weights(state)
  model_weights.assign_weights_to(keras_model)
  results  = keras_model.evaluate(federated_test_data)
  print('\tEval: metrics={}'.format(results))

for round_num in range(2, K):
  keras_evaluate(train_state, round_num)
  result = fed_avg.next(train_state, federated_train_data)
  train_state = result.state
  train_metrics = result.metrics
  print('Round {:2d},  metrics={}'.format(round_num, train_metrics))
  
  #print('\tTrain: loss={l:.3f}, root_mean_squared_error={a:.3f}'.format(
  #    l=train_metrics['loss'], a=train_metrics['root_mean_squared_error']))
  

: 

Training loss is decreasing after each round of federated training, indicating
the model is converging. There are some important caveats with these training
metrics, however, see the section on *Evaluation* later in this tutorial.

## Evaluation

All of our experiments so far presented only federated training metrics - the
average metrics over all batches of data trained across all clients in the
round. This introduces the normal concerns about overfitting, especially since
we used the same set of clients on each round for simplicity, but there is an
additional notion of overfitting in training metrics specific to the Federated
Averaging algorithm. This is easiest to see if we imagine each client had a
single batch of data, and we train on that batch for many iterations (epochs).
In this case, the local model will quickly exactly fit to that one batch, and so
the local accuracy metric we average will approach 1.0. Thus, these training
metrics can be taken as a sign that training is progressing, but not much more.

To perform evaluation on federated data, you can construct another *federated
computation* designed for just this purpose, using the
`tff.learning.build_federated_evaluation` function, and passing in your model
constructor as an argument. Note that unlike with Federated Averaging, where
we've used `MnistTrainableModel`, it suffices to pass the `MnistModel`.
Evaluation doesn't perform gradient descent, and there's no need to construct
optimizers.

For experimentation and research, when a centralized test dataset is available,
[Federated Learning for Text Generation](federated_learning_for_text_generation.ipynb)
demonstrates another evaluation option: taking the trained weights from
federated learning, applying them to a standard Keras model, and then simply
calling `tf.keras.models.Model.evaluate()` on a centralized dataset.